# Rock — FX and ISRC / song / album matching

Reads the combined raw file, attaches FX rates, matches catalog/song keys to the ISRC lookup, runs fallback fills, and writes `Rock_royalties_2021Q1_2026Q1_2_matched.csv`.

Use **Run All**. A timestamped run log is written to the output folder.


In [19]:

import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
from opencc import OpenCC

inputdirectory = '../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/'
globalinputdirectory = '../../50 KM Group/Royalties/Statements/Karen/All labels combined/lookup_tables/'
outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

file_Rock = 'Rock_royalties_2021Q1_2026Q1_1_raw_combined_update.csv'
lookup_fx = 'lookup_tables/Rock_lookup_fx.csv'
lookup_isrc = 'lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx'
outputfile = 'Rock_royalties_2021Q1_2026Q1_2_matched.csv'
converter = OpenCC('s2t')


def resolve_dir(rel):
    cwd = os.getcwd()
    alt = rel.replace('../../', '../', 1) if rel.startswith('../../') else rel
    candidates = [
        os.path.abspath(os.path.join(cwd, rel)),
        os.path.abspath(os.path.join(cwd, 'Karen_statements', rel)),
        os.path.abspath(os.path.join(cwd, alt)),
        os.path.abspath(os.path.join(os.path.dirname(cwd), rel)),
        os.path.abspath(os.path.join(cwd, '..', alt)),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f'Directory not found: {rel}\nTried:\n- ' + '\n- '.join(candidates))


inputdirectory = resolve_dir(inputdirectory)
outputdirectory = resolve_dir(outputdirectory)
try:
    globalinputdirectory = resolve_dir(globalinputdirectory)
except FileNotFoundError:
    globalinputdirectory = None

os.makedirs(outputdirectory, exist_ok=True)
logfile_name = (
    os.path.splitext(outputfile)[0]
    + '_run_log_'
    + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '.txt'
)
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout


class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()

    def flush(self):
        for s in self.streams:
            s.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self.streams[0], name)


sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format


def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f'Run log saved to: {log_path}')


def fmt_int(n):
    try:
        return f'{int(n):,}'
    except (TypeError, ValueError):
        return str(n)


def fmt_money(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_units(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_shape(df):
    return f'{fmt_int(df.shape[0])} rows × {len(df.columns)} columns'


def header(title):
    line = '=' * 72
    print(f'\n{line}\n  {title}\n{line}')


def subheader(title):
    print(f'\n--- {title} ---')


def print_totals(label, df):
    share = df['SHARE AMOUNT (local FX)'].sum() if 'SHARE AMOUNT (local FX)' in df.columns else None
    units = df['UNIT'].sum() if 'UNIT' in df.columns else None
    hkd = df['AMOUNT (HKD)'].sum() if 'AMOUNT (HKD)' in df.columns else None
    line = f'  {label:<28} {fmt_shape(df)}'
    if share is not None:
        line += f'    SHARE {fmt_money(share):>16}'
    if hkd is not None:
        line += f'    AMOUNT (HKD) {fmt_money(hkd):>14}'
    if units is not None:
        line += f'    UNIT {fmt_units(units):>18}'
    print(line)


def readfile(directory, file):
    path = os.path.join(directory, file)
    df = pd.read_csv(path, low_memory=False)
    print(f'  CSV    {fmt_shape(df):<32}  {file}')
    return df


def readfilexls(directory, file, sheet):
    path = os.path.join(directory, file)
    df = pd.read_excel(path, sheet_name=sheet)
    print(f'  Excel  {fmt_shape(df):<32}  {file} / {sheet}')
    return df


def merge(df1, df2, col, what):
    subheader(f"Merge '{what}' on '{col}'")
    empty_cells = df1[col].isna().sum()
    print(f"  Empty '{col}' before merge : {fmt_int(empty_cells)}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    df_merged = pd.merge(df1, df2, on=col, how='left', indicator='_merge_status')
    unmatched_mask = df_merged['_merge_status'] == 'left_only'
    n_unmatched = unmatched_mask.sum()
    df_merged = df_merged.drop(columns='_merge_status')
    col_safe = col.replace('/', '')
    if n_unmatched == 0:
        print('  Match result             : successful')
    else:
        issue_file = f'match_issues_{what}_{col_safe}.csv'
        print(f'  Match result             : {fmt_int(n_unmatched)} unmatched rows  -> {issue_file}')
        empty_rows = df_merged.loc[unmatched_mask]
        empty_rows.to_csv(os.path.join(outputdirectory, issue_file), index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_file = f'unused_lookup_rows_{col_safe}.csv'
        unused_rows.to_csv(os.path.join(outputdirectory, unused_file), index=False)
        print(f'  Unused lookup rows       : {fmt_int(len(unused_rows))}  -> {unused_file}')
    else:
        print('  Unused lookup rows       : 0')
    print(f'  After merge              : {fmt_shape(df_merged)}')
    return df_merged


last_match_issues_path = None


def merge_and_return_empty_rows(df1, df2, col, what):
    global last_match_issues_path
    last_match_issues_path = None
    subheader(f"Merge '{what}' on '{col}'")
    empty_cells = df1[col].isna().sum()
    print(f"  Empty '{col}' before merge : {fmt_int(empty_cells)}")
    df1.loc[:, col] = df1[col].fillna('XX_UNKNOWN')
    df_merged = pd.merge(df1, df2, on=col, how='left', indicator='_merge_status')
    unmatched_mask = df_merged['_merge_status'] == 'left_only'
    n_unmatched = int(unmatched_mask.sum())
    empty_rows = df_merged.loc[unmatched_mask].drop(columns='_merge_status')
    df_merged = df_merged.drop(columns='_merge_status')
    col_safe = col.replace('/', '')
    if n_unmatched == 0:
        print('  Match result             : successful')
    else:
        issue_file = f'match_issues_{what}_{col_safe}.csv'
        last_match_issues_path = os.path.join(outputdirectory, issue_file)
        print(f'  Match result             : {fmt_int(n_unmatched)} unmatched rows  -> {issue_file}')
        empty_rows.to_csv(last_match_issues_path, index=False)
    unused_rows = df2[~df2[col].isin(df_merged[col])]
    if len(unused_rows) > 0:
        unused_file = f'unused_lookup_rows_{col_safe}.csv'
        unused_rows.to_csv(os.path.join(outputdirectory, unused_file), index=False)
        print(f'  Unused lookup rows       : {fmt_int(len(unused_rows))}  -> {unused_file}')
    else:
        print('  Unused lookup rows       : 0')
    print(f'  After merge              : {fmt_shape(df_merged)}')
    return df_merged, empty_rows


header('Rock — matching')
print(f'  Run started              : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Input dir                : {inputdirectory}')
print(f'  Output dir               : {outputdirectory}')
print(f'  Output file              : {outputfile}')
print(f'  Run log                  : {logfile_name}')

header('1. Required files')
required = [
    os.path.join(inputdirectory, file_Rock),
    os.path.join(inputdirectory, lookup_fx),
    os.path.join(inputdirectory, lookup_isrc),
]
missing = []
for p in required:
    if os.path.isfile(p):
        print(f'  [OK]       {p}')
    else:
        print(f'  [MISSING]  {p}')
        missing.append(p)
if missing:
    close_log()
    raise FileNotFoundError('Stopping: required file(s) not found:\n- ' + '\n- '.join(missing))

header('2. Load files')
df_Rock = readfile(inputdirectory, file_Rock)
print_totals('Raw combined', df_Rock)
df_lookup_fx = readfile(inputdirectory, lookup_fx)
df_lookup_isrc = readfilexls(inputdirectory, lookup_isrc, 'Data_MOD')



  Rock — matching
  Run started              : 2026-08-19 17:34:32
  Input dir                : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements
  Output dir               : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/_output
  Output file              : Rock_royalties_2021Q1_2026Q1_2_matched.csv
  Run log                  : Rock_royalties_2021Q1_2026Q1_2_matched_run_log_20260819_173432.txt

  1. Required files
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_1_raw_combined_update.csv
  [OK]       /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/lookup_tables/Rock_lookup_fx.csv
  [OK]       /Users/johannesnatter

In [20]:

header('3. Attach FX rates')
print('  Merge keys                : Report year, Report Quarter, Currency')
df_Rock_fx = pd.merge(
    df_Rock,
    df_lookup_fx[['Report year', 'Report Quarter', 'Currency', 'FX rate']],
    on=['Report year', 'Report Quarter', 'Currency'],
    how='left',
)
n_fx_missing = df_Rock_fx['FX rate'].isna().sum()
print(f'  Rows missing FX rate     : {fmt_int(n_fx_missing)}')
if n_fx_missing:
    print('  ALERT: some rows have no FX rate')

df_Rock_fx['SHARE AMOUNT (local FX)'] = pd.to_numeric(df_Rock_fx['SHARE AMOUNT (local FX)'], errors='coerce')
df_Rock_fx['AMOUNT (HKD)'] = df_Rock_fx['SHARE AMOUNT (local FX)'] / df_Rock_fx['FX rate']
df_Rock_fx['Amount to Rock (HKD)'] = np.where(
    df_Rock_fx['AMOUNT (HKD)'] != 0,
    df_Rock_fx['AMOUNT'] * df_Rock_fx['AMOUNT (HKD)'] / df_Rock_fx['SHARE AMOUNT (local FX)'],
    0,
)

df_Rock_fx.rename(columns={'FX rate': 'FX rate_main'}, inplace=True)
df_lookup_fx_rmb = df_lookup_fx[df_lookup_fx['Currency'] == 'RMB']
df_Rock_fx = pd.merge(
    df_Rock_fx,
    df_lookup_fx_rmb[['Report year', 'Report Quarter', 'FX rate']],
    on=['Report year', 'Report Quarter'],
    how='left',
)
n_rmb_missing = df_Rock_fx['FX rate'].isna().sum()
print(f'  Rows missing RMB FX      : {fmt_int(n_rmb_missing)}')
df_Rock_fx.rename(columns={'FX rate': 'FX rate_rmb', 'FX rate_main': 'FX rate'}, inplace=True)
df_Rock_fx['Amount to Rock (RMB)'] = df_Rock_fx['Amount to Rock (HKD)'] * df_Rock_fx['FX rate_rmb']

empty_cells_p = df_Rock_fx['USER'].isna().sum()
empty_cells_a = df_Rock_fx['CATALOG TITLE'].isna().sum()
empty_cells_s = df_Rock_fx['SONG TITLE'].isna().sum()
print(
    f"  Empty USER / CATALOG TITLE / SONG TITLE : "
    f'{fmt_int(empty_cells_p)} / {fmt_int(empty_cells_a)} / {fmt_int(empty_cells_s)}'
)
df_Rock_fx.fillna({'USER': 'XX_UNKNOWN', 'CATALOG TITLE': 'Song Type', 'SONG TITLE': 'EMPTY'}, inplace=True)
print_totals('After FX', df_Rock_fx)



  3. Attach FX rates
  Merge keys                : Report year, Report Quarter, Currency
  Rows missing FX rate     : 0
  Rows missing RMB FX      : 0
  Empty USER / CATALOG TITLE / SONG TITLE : 0 / 0 / 455
  After FX                     1,442,752 rows × 34 columns    SHARE     6,893,726.36    AMOUNT (HKD)   3,951,607.08    UNIT   7,354,374,169.00


In [21]:

header('4. Match catalog / title / song')
df_Rock_fx['CATALOG NO.'] = df_Rock_fx['CATALOG NO.'].astype('string')
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'] = (
    df_Rock_fx['CATALOG NO._MOD'] + '.' + df_Rock_fx['CATALOG TITLE'] + '.' + df_Rock_fx['SONG TITLE']
)
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'] = df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE'].astype(str)
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = (
    df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE']
    .str.replace(' ', '', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('-', '', regex=False)
    .str.replace('/', '', regex=False)
    .str.replace('（', '', regex=False)
    .str.replace('）', '', regex=False)
    .str.replace('’', '', regex=False)
    .str.replace("'", '', regex=False)
    .str.lower()
)
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'].apply(
    converter.convert
)
df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'] = (
    df_Rock_fx['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'].str.replace('.empty', '.', regex=False)
)

df_Rock_matched, unmapped_rows = merge_and_return_empty_rows(
    df_Rock_fx, df_lookup_isrc, 'CATALOG NO..CATALOG TITLE.SONG TITLE_MOD', 'Rock_fx'
)
print(f'  Unmapped rows for fallback : {fmt_int(len(unmapped_rows))}')

df_Rock_matched = df_Rock_matched.drop(columns=['CATALOG NO..CATALOG TITLE.SONG TITLE'])
df_Rock_matched = df_Rock_matched.drop(columns=['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
df_Rock_matched = df_Rock_matched.drop(columns=['CATALOG NO._MOD'])
df_Rock_matched = df_Rock_matched.sort_index(axis=1)
print_totals('After catalog match', df_Rock_matched)



  4. Match catalog / title / song

--- Merge 'Rock_fx' on 'CATALOG NO..CATALOG TITLE.SONG TITLE_MOD' ---
  Empty 'CATALOG NO..CATALOG TITLE.SONG TITLE_MOD' before merge : 0
  Match result             : 34,240 unmatched rows  -> match_issues_Rock_fx_CATALOG NO..CATALOG TITLE.SONG TITLE_MOD.csv
  Unused lookup rows       : 0
  After merge              : 1,442,752 rows × 40 columns
  Unmapped rows for fallback : 34,240
  After catalog match          1,442,752 rows × 37 columns    SHARE     6,893,726.36    AMOUNT (HKD)   3,951,607.08    UNIT   7,354,374,169.00


In [22]:

header('5. Load ISRC lookup dictionaries')
all_isrc = readfilexls(inputdirectory, lookup_isrc, 'lookup')


def normalize_catalog_title(title):
    s = (
        str(title)
        .replace(' ', '')
        .replace('(', '')
        .replace(')', '')
        .replace('-', '')
        .replace('/', '')
        .replace('（', '')
        .replace('）', '')
        .replace('’', '')
        .replace('《', '')
        .replace('》', '')
        .replace('`', '')
        .replace("'", '')
    )
    return converter.convert(s.lower())


isrc_to_song = {}
song_to_isrc = {}
song_norm_to_isrc = {}
song_norm_to_song = {}
for index, row in all_isrc.iterrows():
    isrc = row['ISRC']
    song = row['Song (final)']
    isrc_to_song[isrc] = song
    song_to_isrc[song] = isrc
    norm_song = normalize_catalog_title(song)
    song_norm_to_isrc[norm_song] = isrc
    song_norm_to_song[norm_song] = song

print(f'  ISRC lookup songs        : {fmt_int(len(isrc_to_song))}')

all_titles = readfilexls(inputdirectory, lookup_isrc, 'Data_MOD')

title_to_album = {}
title_to_albumtype = {}
album_to_albumtype = {}


def extract_title_key(mod_string):
    parts = str(mod_string).split('.')
    if len(parts) < 3:
        raw_title = str(mod_string)
    else:
        raw_title = '.'.join(parts[2:-1])
    return normalize_catalog_title(raw_title)[:10]


for index, row in all_titles.iterrows():
    title = extract_title_key(row['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
    album = row['Album']
    albumtype = row['Album Type']
    title_to_album[title] = album
    title_to_albumtype[title] = albumtype
    album_to_albumtype[album] = albumtype

print(f'  Title keys for album     : {fmt_int(len(title_to_album))}')



  5. Load ISRC lookup dictionaries
  Excel  313 rows × 6 columns              lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx / lookup
  ISRC lookup songs        : 313
  Excel  5,243 rows × 5 columns            lookup_tables/Rock_lookup_ISRC_song_album_albumtype.xlsx / Data_MOD
  Title keys for album     : 582


In [23]:

header('6. Fallback fills on unmatched rows')

UNFILLED_MARKERS = {'XX_UNKNOWN', 'XX_UNKNOWN'}


def count_filled(before_df, after_df, columns=('ISRC', 'Song')):
    """Count cells filled by the algorithm (empty before, real value after)."""
    cols = list(columns)
    was_empty = before_df[cols].isna() | before_df[cols].isin(UNFILLED_MARKERS)
    total_rows = was_empty.any(axis=1).sum()
    now_filled = after_df[cols].notna() & ~after_df[cols].isin(UNFILLED_MARKERS)
    filled = was_empty & now_filled
    per_column = filled.sum()
    rows_with_any = filled.any(axis=1).sum()
    print(f'  Unmapped rows            : {fmt_int(total_rows)}')
    print('  Values filled:')
    for col in cols:
        pct = (per_column[col] / total_rows * 100) if total_rows else 0
        print(f'    {col:<22} {fmt_int(per_column[col]):>10}  ({pct:.1f}%)')
    rows_pct = (rows_with_any / total_rows * 100) if total_rows else 0
    print(f'  Rows with any filled     : {fmt_int(rows_with_any)}  ({rows_pct:.1f}%)')
    return per_column, rows_with_any


def normalize_for_match(text):
    s = str(text).lower()
    for ch in " `'\"()-（）[]":
        s = s.replace(ch, '')
    return s


def find_subsequence_match(text, candidates, normalizer=None):
    """Return the first candidate whose characters appear in order within text."""
    norm_fn = normalizer or (lambda x: str(x).lower())
    char_list = list(norm_fn(text))
    for candidate in candidates:
        idx = 0
        matched = True
        for char in norm_fn(candidate):
            while idx < len(char_list) and char_list[idx] != char:
                idx += 1
            if idx >= len(char_list):
                matched = False
                break
            idx += 1
        if matched:
            return candidate
    return None


def find_isrc_for_song(matched_song, song_to_isrc):
    """Find ISRC for a matched song via exact, forward, or reverse subsequence match."""
    if matched_song in song_to_isrc:
        return song_to_isrc[matched_song]
    lookup_song = find_subsequence_match(matched_song, song_to_isrc.keys(), normalize_for_match)
    if lookup_song is not None:
        return song_to_isrc[lookup_song]
    for lookup_song in song_to_isrc:
        if find_subsequence_match(lookup_song, [matched_song], normalize_for_match):
            return song_to_isrc[lookup_song]
    return None


def fill_rows_album(df):
    rows_filled = df.copy()
    rows_filled['Album'] = rows_filled['Album'].astype('object')
    rows_filled['Album Type'] = rows_filled['Album Type'].astype('object')
    for index, row in rows_filled.iterrows():
        title_key = extract_title_key(row['CATALOG NO..CATALOG TITLE.SONG TITLE_MOD'])
        album_empty = pd.isna(rows_filled.at[index, 'Album']) or rows_filled.at[index, 'Album'] == 'XX_UNKNOWN'
        albumtype_empty = pd.isna(rows_filled.at[index, 'Album Type']) or rows_filled.at[index, 'Album Type'] == 'XX_UNKNOWN'
        if album_empty and title_key in title_to_album:
            rows_filled.at[index, 'Album'] = title_to_album[title_key]
        if albumtype_empty and title_key in title_to_albumtype:
            rows_filled.at[index, 'Album Type'] = title_to_albumtype[title_key]
    return rows_filled


def fill_rows_ISRC_song(df):
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        isrc_number = row['CATALOG NO.']
        if isrc_number in isrc_to_song:
            song_name = isrc_to_song[isrc_number]
            rows_filled.at[index, 'ISRC'] = isrc_number
            rows_filled.at[index, 'Song'] = song_name
        else:
            song = row['SONG TITLE']
            if song in song_to_isrc:
                rows_filled.at[index, 'ISRC'] = song_to_isrc[song]
                rows_filled.at[index, 'Song'] = song
            else:
                norm_song = normalize_catalog_title(song)
                if norm_song in song_norm_to_isrc:
                    rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
                    rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
                else:
                    rows_filled.at[index, 'Song'] = 'XX_UNKNOWN'
                    rows_filled.at[index, 'ISRC'] = 'XX_UNKNOWN'
    return rows_filled


adapted_df = fill_rows_ISRC_song(unmapped_rows)
subheader('ISRC / Song fill')
count_filled(unmapped_rows, adapted_df)

adapted_df = fill_rows_album(adapted_df)
subheader('Album / Album Type fill')
count_filled(unmapped_rows, adapted_df, columns=('Album', 'Album Type'))


def fill_rows_ISRC_song_ignore_lyric_mv(df):
    """Attempt: ignore a '歌詞版mv' (lyric video) suffix in the song title before matching."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        norm_song = normalize_catalog_title(row['SONG TITLE']).replace('歌詞版mv', '')
        if norm_song in song_norm_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
            rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
    return rows_filled


before_lyric_mv_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_ignore_lyric_mv(adapted_df)
subheader('Lyric-video (歌詞版MV) suffix pass')
count_filled(before_lyric_mv_pass, adapted_df, columns=('ISRC', 'Song'))

song_norm7_to_isrc = {}
song_norm7_to_song = {}
for isrc, song in isrc_to_song.items():
    key7 = normalize_catalog_title(song)[:7]
    song_norm7_to_isrc[key7] = isrc
    song_norm7_to_song[key7] = song


def fill_rows_ISRC_song_7char(df):
    """Second-pass attempt: match on the first 7 characters of the normalized song title."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        key7 = normalize_catalog_title(row['SONG TITLE'])[:7]
        if key7 in song_norm7_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm7_to_isrc[key7]
            rows_filled.at[index, 'Song'] = song_norm7_to_song[key7]
    return rows_filled


before_7char_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_7char(adapted_df)
subheader('First 7 characters of normalized song title')
count_filled(before_7char_pass, adapted_df, columns=('ISRC', 'Song'))


def fill_rows_ISRC_song_ignore_cantonese(df):
    """Attempt: ignore the '粵' (Cantonese) marker in the song title before matching."""
    rows_filled = df.copy()
    for index, row in rows_filled.iterrows():
        if row['ISRC'] != 'XX_UNKNOWN':
            continue
        norm_song = normalize_catalog_title(row['SONG TITLE']).replace('粵', '')
        if norm_song in song_norm_to_isrc:
            rows_filled.at[index, 'ISRC'] = song_norm_to_isrc[norm_song]
            rows_filled.at[index, 'Song'] = song_norm_to_song[norm_song]
    return rows_filled


before_cantonese_pass = adapted_df.copy()
adapted_df = fill_rows_ISRC_song_ignore_cantonese(adapted_df)
subheader('Cantonese marker (粵) pass')
count_filled(before_cantonese_pass, adapted_df, columns=('ISRC', 'Song'))

check_cols = ['ISRC', 'Song', 'Album', 'Album Type']
unfilled_mask = adapted_df[check_cols].isna() | adapted_df[check_cols].isin(UNFILLED_MARKERS)
rows_still_unfilled = adapted_df[unfilled_mask.any(axis=1)]
print()
print(f'  Still unfilled in {check_cols} : {fmt_int(len(rows_still_unfilled))}')

if len(rows_still_unfilled) == 0:
    if last_match_issues_path and os.path.isfile(last_match_issues_path):
        os.remove(last_match_issues_path)
        print(f'  Deleted resolved issues  : {os.path.basename(last_match_issues_path)}')
    else:
        print('  Match-issues file        : none to delete')
else:
    unfilled_outputfile = 'Rock_royalties_2021Q1_2026Q1_2_unfilled_rows.xlsx'
    unfilled_path = os.path.join(outputdirectory, unfilled_outputfile)
    rows_still_unfilled.to_excel(unfilled_path, index=False, engine='openpyxl')
    print(f'  Wrote unfilled rows      : {fmt_int(len(rows_still_unfilled))}  -> {unfilled_path}')



  6. Fallback fills on unmatched rows

--- ISRC / Song fill ---
  Unmapped rows            : 34,240
  Values filled:
    ISRC                       33,501  (97.8%)
    Song                       33,501  (97.8%)
  Rows with any filled     : 33,501  (97.8%)

--- Album / Album Type fill ---
  Unmapped rows            : 34,240
  Values filled:
    Album                      34,240  (100.0%)
    Album Type                 34,240  (100.0%)
  Rows with any filled     : 34,240  (100.0%)

--- Lyric-video (歌詞版MV) suffix pass ---
  Unmapped rows            : 739
  Values filled:
    ISRC                          144  (19.5%)
    Song                          144  (19.5%)
  Rows with any filled     : 144  (19.5%)

--- First 7 characters of normalized song title ---
  Unmapped rows            : 595
  Values filled:
    ISRC                          362  (60.8%)
    Song                          362  (60.8%)
  Rows with any filled     : 362  (60.8%)

--- Cantonese marker (粵) pass ---
  Unmapped row

In [24]:

header('7. Save matched file')


def is_problem_value(series):
    as_str = series.astype(str).str.strip()
    return (
        series.isna()
        | as_str.eq('')
        | as_str.str.upper().eq('NAN')
        | as_str.eq('XX_UNKNOWN')
    )


check_cols_final = ['ISRC', 'Song', 'Album', 'Album Type']
problem_mask = adapted_df[check_cols_final].apply(is_problem_value)
rows_with_problems = adapted_df[problem_mask.any(axis=1)]
cols_with_problems = [col for col in check_cols_final if problem_mask[col].any()]
print(f'  Fallback rows with XX_UNKNOWN / blank / NAN : {fmt_int(len(rows_with_problems))}')
print(f'  Columns affected             : {cols_with_problems if cols_with_problems else "none"}')
for col in check_cols_final:
    print(f'    {col:<22} {fmt_int(problem_mask[col].sum())}')

fill_cols = ['ISRC', 'Song', 'Album', 'Album Type']
df_Rock_matched.loc[adapted_df.index, fill_cols] = adapted_df[fill_cols]
df_Rock_matched.fillna({'Song': 'EMPTY', 'Album': 'XX_UNKNOWN', 'Album Type': 'XX_UNKNOWN'}, inplace=True)

print()
print_totals('Final matched', df_Rock_matched)
print()
print(f'  Output columns ({len(df_Rock_matched.columns)}):')
print('    ' + ', '.join(df_Rock_matched.columns.astype(str)))

path = os.path.join(outputdirectory, outputfile)
df_Rock_matched.to_csv(path, index=False)
print()
print(f'  CSV written              : {path}')
print(f'  {fmt_shape(df_Rock_matched)}')
print(f'  Run finished             : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
close_log()



  7. Save matched file
  Fallback rows with XX_UNKNOWN / blank / NAN : 0
  Columns affected             : none
    ISRC                   0
    Song                   0
    Album                  0
    Album Type             0

  Final matched                1,442,752 rows × 37 columns    SHARE     6,893,726.36    AMOUNT (HKD)   3,951,607.08    UNIT   7,354,374,169.00

  Output columns (37):
    AMOUNT, AMOUNT (HKD), ARTIST, Album, Album Type, Amount to Rock (HKD), Amount to Rock (RMB), Base Price, CATALOG NO., CATALOG TITLE, Ctrl.%, Currency, Entry No., FX rate, FX rate_rmb, ISRC, PayType, Payee/Licensor, Payer/Licensee, REVENUE PERIOD, ROYALTY, Release Date, Report Quarter, Report year, Rev Quarter, Rev Year, Royalty Rate%, SHARE AMOUNT (local FX), SONG TITLE, Share%, Song, SongProRata, Territory, Type, UNIT, USER, WS Price

  CSV written              : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_20